In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer

In [ ]:
# Configuration

CSV_PATH = "posts_for_topic_modelling.csv"
TEXT_COL = "post_text"   
ID_COL = "new_post_id"                            
MODEL_NAME = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
OUTPUT_EMBEDDINGS = "post_embeddings.npy"
OUTPUT_INSPECTION_PLOT = "post_length_inspection.png"
OUTPUT_CLEAN_CSV = "posts_clean.csv"

In [ ]:
# Loading data

print("Loading data...")
df_raw = pd.read_csv(CSV_PATH)
n_before = len(df_raw)
 
df = df_raw[df_raw[TEXT_COL].notna()].reset_index(drop=True)
n_after = len(df)
n_dropped = n_before - n_after
 
posts = df[TEXT_COL].astype(str).tolist()
print(f"Number of posts in CSV file: {n_before:,}")
if n_dropped > 0:
    print(f"Of these, excluded due to missing post_text (NaN/empty string): {n_dropped:,}")
    print("(The Stata export excluded empty `post_text` values, but a few were "
          "likely exported as an empty string, which pandas interprets as NaN "
          "upon loading—consequently, they are only caught here.)")
print(f"Number of posts for embedding: {n_after:,}")

# Number of records in the CSV file: 46,359
# Number of records for embedding: 46,346
# 13 records had the value NaN; these were not detected as missing in Stata, but were detected by pandas here. They are excluded.

In [ ]:
# Loading model

print(f"\nLoading model: {MODEL_NAME}")
model = SentenceTransformer(MODEL_NAME)
tokenizer = model.tokenizer
max_seq_length = model.max_seq_length
print(f"Models max_seq_length: {max_seq_length} tokens")
print("(Text beyond this will be automatically truncated by the model.)")

In [ ]:
# Inspecting posts

print("\nCalculates length for each post...")
 
char_lengths = [len(p) for p in posts]
word_counts = [len(p.split()) for p in posts]
 
# Tokenize all posts 
token_counts = [
    len(tokenizer.encode(p, add_special_tokens=True, truncation=False))
    for p in posts
]
 
df_inspect = pd.DataFrame({
    "char_length": char_lengths,
    "word_count": word_counts,
    "token_count": token_counts,
})
 
print("\n- Descriptive statistic -")
print(df_inspect.describe(percentiles=[.5, .75, .9, .95, .99]).round(1))
 
n_above_limit = sum(t > max_seq_length for t in token_counts)
pct_above_limit = 100 * n_above_limit / len(token_counts)
print(f"\nPosts exceeding max_seq_length ({max_seq_length} tokens): "
      f"{n_above_limit:,} out of {len(token_counts):,} ({pct_above_limit:.2f}%)")

In [ ]:
# Generate embeddings

print("\nGenerates embeddings on raw posts (no preprocessing applied)...")
embeddings = model.encode(
    posts,
    show_progress_bar=True,
    batch_size=64,
    convert_to_numpy=True,
)
print(f"Embeddings generated with shape: {embeddings.shape}")

np.save(OUTPUT_EMBEDDINGS, embeddings)
print(f"Embeddings saved as: {OUTPUT_EMBEDDINGS}")

df_inspect_out = pd.concat([df[[ID_COL, TEXT_COL]], df_inspect], axis=1)
df_inspect_out.to_csv("posts_clean.csv", index=False)
print(f"Cleaned dataset saved as: {OUTPUT_CLEAN_CSV}")

print("\nDone! Embeddings are ready for BERTopic:")
print("    from bertopic import BERTopic")
print("    topic_model = BERTopic(language='multilingual')")
print("    topics, probs = topic_model.fit_transform(posts, embeddings=embeddings)")